In [35]:
import os
from neo4j import GraphDatabase
from yfiles_jupyter_graphs import GraphWidget
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough, ConfigurableField
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts.prompt import PromptTemplate
from langchain_core.pydantic_v1 import BaseModel, Field
from typing import List
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.graphs import Neo4jGraph
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_community.vectorstores import Neo4jVector
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores.neo4j_vector import remove_lucene_chars

In [42]:
from llama_cpp import Llama

In [13]:
from modules import logging

In [33]:
logging.langsmith("PDF-RAG")

LangSmith 추적을 시작합니다.
[프로젝트명]
PDF-RAG


In [14]:
# API KEY를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API KEY 정보로드
load_dotenv()

True

In [15]:
def show_metadata(docs):
    if docs:
        print("[metadata]")
        print(list(docs[0].metadata.keys()))
        print("\n[examples]")
        max_key_length = max(len(k) for k in docs[0].metadata.keys())
        for k, v in docs[0].metadata.items():
            print(f"{k:<{max_key_length}} : {v}")

In [16]:
import getpass

# os.environ["OPENAI_API_KEY"] = getpass.getpass(prompt = 'OpenAIのAPIキーを入力してください')
os.environ["NEO4J_URI"] = getpass.getpass(prompt = 'neo4j+s://db8bf133.databases.neo4j.io')
os.environ["NEO4J_USERNAME"] = getpass.getpass(prompt = 'neo4j')
os.environ["NEO4J_PASSWORD"] = getpass.getpass(prompt = 'gTCqLY75bwR5rz5Fc8127o8DFkwkmfErDgIg_-ndiPM')

In [23]:

FILE_PATH = "data\자동차 관리 _ 손해배상 보장 법.pdf"

In [27]:
raw_documents = PyPDFLoader(FILE_PATH).load()

In [ ]:
print(raw_documents[5].page_content[:1000])

In [31]:
show_metadata(raw_documents)

[metadata]
['source', 'page']

[examples]
source : data\자동차 관리 _ 손해배상 보장 법.pdf
page   : 0


In [36]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=125)

In [37]:
documents = text_splitter.split_documents(raw_documents)

In [38]:
print(f"분할된 청크의수: {len(documents)}")

분할된 청크의수: 814
